<a href="https://colab.research.google.com/github/europecodingschool/ECS---VB---YZ---08.26/blob/main/VB_YZ_Ders_12_3_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB, GaussianNB

In [6]:
sms = pd.read_csv("/content/sms_mesajlari.csv")

In [7]:
sms.head()

,mesaj_id,mesaj,etiket
0,1,Hesabınıza 2.750 TL maaş ödemesi yatırılmıştır...,normal
1,2,Kartınızdan 2.750 TL harcama yapılmıştır. Size,dolandiricilik
2,3,Hesabınıza 640 TL maaş ödemesi yatırılmıştır. ...,normal
3,4,"Toplantı notlarını mail attım, bakabilir misin...",normal
4,5,Randevunuz pazartesi saat 16:45 olarak onaylan...,normal


Senaryo: Gelen bir SMS'in dolandırıcılık olup olmadığını, sadece metne bakarak tahmin edebilir miyiz?

In [9]:
sms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   mesaj_id  1500 non-null   int64 
 1   mesaj     1500 non-null   object
 2   etiket    1500 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.3+ KB


In [11]:
sms.shape

(1500, 3)

In [13]:
sms['etiket'].value_counts()

,count
etiket,
normal,1200
dolandiricilik,300


In [15]:
sms['etiket'].value_counts(normalize=True)

,proportion
etiket,
normal,0.8
dolandiricilik,0.2


In [19]:
dolandirici = sms[sms['etiket'] == "dolandiricilik"] # bir for döngüsü yazıp csv dosyası içindeki dolandırıcılık mesaklarını bir bakmak istedik. 5 tane getirmesini istedik.
normal = sms[sms['etiket'] == "normal"]
print("****Dolandırıcı Mesaj Örnekleri******")
for mesaj in dolandirici['mesaj'].sample(5):
  print(mesaj)


****Dolandırıcı Mesaj Örnekleri******
Paketiniz gümrükte bekliyor, 24 TL ödeme yapmazsanız iade edilecek: edevlet-iade.info/a945
Elif ben, yeni numaram bu. Acil para lazım, IBAN atıyorum.
Siparişiniz kurye ile yola çıktı. Teslimat için onay verin: hesap-dogrula.com/a879 Detaylar için arayın.
Randevunuz iptal edilmiştir. Yeniden onay için 5.000 TL ödeme yapınız. Bilgilerinize.
Hesabınız bloke edilmiştir. Açmak için hemen tıklayın: edevlet-iade.info/a513


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

In [23]:
deneme = [
    "Hesabınız Bloke edildi hemen tıklayın",
    "Maaş Ödemeniz hesabınıza yapıldı",
    "Hemen tıklayın hemen kazanın",
]

deneme_sayici = CountVectorizer()
deneme_sayici.fit(deneme)

CountVectorizer()



*   Harfleri Küçülttü.
*   Mesajları kelimeler böldü.
*   Tekrarları bir kez yazdı
*.  Alfabetik sıraladı



In [25]:
matris = deneme_sayici.transform(deneme)
tablo = pd.DataFrame(matris.toarray(), columns=deneme_sayici.get_feature_names_out(),
                     index=["mesaj1","mesaj2","mesaj3"])
print(tablo)

        bloke  edildi  hemen  hesabınız  hesabınıza  kazanın  maaş  tıklayın  \
mesaj1      1       1      1          1           0        0     0         1   
mesaj2      0       0      0          0           1        0     1         0   
mesaj3      0       0      2          0           0        1     0         1   

        yapıldı  ödemeniz  
mesaj1        0         0  
mesaj2        1         1  
mesaj3        0         0  


In [27]:
#1. Veriyi Ayırıma
X = sms['mesaj']
y = sms['etiket']

In [33]:
#2. Train ve Test olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [34]:
print("Eğitim datası sms adeti:", len(X_train))
print("Test datası sms adeti:", len(X_test))

Eğitim datası sms adeti: 1125
Test datası sms adeti: 375


In [36]:
# 3.SMS metinlerini vektörize ettik. Algoritmanın anlayabileceği hale getirdik.
sayici = CountVectorizer()
X_train_vec = sayici.fit_transform(X_train)
X_test_vec = sayici.transform(X_test)

In [39]:
print("Eğitim Tablosu:", X_train_vec.shape)
print("Test Tablosu:", X_test_vec.shape)

Eğitim Tablosu: (1125, 476)
Test Tablosu: (375, 476)


In [43]:
#4.Model Eğitimi
model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

In [45]:
#5. Tahminleme
tahmin = model.predict(X_test_vec)

In [47]:
#6. Skorlarımıza baktık
accuracy_score(y_test, tahmin)

0.9866666666666667

In [49]:
confusion_matrix(y_test, tahmin)

array([[ 75,   0],
       [  5, 295]])